In [ ]:
"""
THIS CODE MOSTLY WILL NOT FIND DATA BECAUSE FLIPKART AND AMAZON USES SOME ANTI SCRAPING PROTECTIONS AND THEY KEEP CHANGING THE HTML
"""


import requests
from bs4 import BeautifulSoup
import csv

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/117.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
}

# ---------- Flipkart Scraper ----------
def scrape_flipkart(search_query):
    query = search_query.replace(" ", "+")
    url = f"https://www.flipkart.com/search?q={query}"
    response = requests.get(url, headers=HEADERS)
    products = []

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")

        for item in soup.find_all("div", {"class": "_1AtVbE"}):
            # Flipkart product selectors (varies by category)
            title = item.find("div", {"class": "_4rR01T"}) or item.find("a", {"class": "s1Q9rs"})
            price = item.find("div", {"class": "_30jeq3"})
            rating = item.find("div", {"class": "_3LWZlK"})
            link = item.find("a", {"class": "_1fQZEK"}) or item.find("a", {"class": "s1Q9rs"})

            if title and price and link:
                products.append({
                    "Platform": "Flipkart",
                    "Title": title.text.strip(),
                    "Price": price.text.strip(),
                    "Rating": rating.text.strip() if rating else "N/A",
                    "Link": "https://www.flipkart.com" + link["href"]
                })
    return products


# ---------- Amazon Scraper ----------
def scrape_amazon(search_query):
    query = search_query.replace(" ", "+")
    url = f"https://www.amazon.in/s?k={query}"
    response = requests.get(url, headers=HEADERS)
    products = []

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")

        for item in soup.find_all("div", {"data-component-type": "s-search-result"}):
            title_tag = item.h2.a.span if item.h2 and item.h2.a and item.h2.a.span else None
            title = title_tag.text.strip() if title_tag else None

            price_tag = item.find("span", {"class": "a-price-whole"})
            price = "₹" + price_tag.text.strip() if price_tag else "N/A"

            rating_tag = item.find("span", {"class": "a-icon-alt"})
            rating = rating_tag.text.strip() if rating_tag else "N/A"

            link_tag = item.h2.a["href"] if item.h2 and item.h2.a else None
            link = "https://www.amazon.in" + link_tag if link_tag else "N/A"

            if title:
                products.append({
                    "Platform": "Amazon",
                    "Title": title,
                    "Price": price,
                    "Rating": rating,
                    "Link": link
                })
    return products


# ---------- Save to CSV ----------
def save_to_csv(products, filename="products_comparison.csv"):
    with open(filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["Platform", "Title", "Price", "Rating", "Link"])
        writer.writeheader()
        writer.writerows(products)
    print(f"✅ Data saved to {filename}")


# ---------- Main Program ----------
if __name__ == "__main__":
    query = input("🔍 Enter product to search: ")

    print("\n⏳ Scraping Flipkart...")
    flipkart_results = scrape_flipkart(query)

    print("⏳ Scraping Amazon...")
    amazon_results = scrape_amazon(query)

    all_results = flipkart_results + amazon_results

    if all_results:
        print(f"\n✅ Found {len(all_results)} products\n")

        for idx, p in enumerate(all_results, start=1):
            print(f"{idx}. [{p['Platform']}] {p['Title']}")
            print(f"   Price  : {p['Price']}")
            print(f"   Rating : {p['Rating']}")
            print(f"   Link   : {p['Link']}\n")

        save_to_csv(all_results)
    else:
        print("⚠ No products found on Flipkart or Amazon.")



⏳ Scraping Flipkart...
⏳ Scraping Amazon...
⚠ No products found on Flipkart or Amazon.


In [2]:
# books_scraper.py
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import csv
import time

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/117.0.0.0 Safari/537.36"),
    "Accept-Language": "en-US,en;q=0.9"
}

def rating_to_int(class_list):
    # 'star-rating Three' -> 3
    mapping = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    for c in class_list:
        if c in mapping:
            return mapping[c]
    return None

def scrape_books(base_url="http://books.toscrape.com/", max_pages=None, delay=0.8, output_csv="books.csv"):
    url = base_url
    page = 1
    results = []

    while True:
        print(f"Scraping page {page}: {url}")
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # Each product is in article.product_pod
        for item in soup.select("article.product_pod"):
            try:
                title_tag = item.h3.a
                title = title_tag["title"].strip() if title_tag and title_tag.has_attr("title") else title_tag.text.strip()

                rel_link = title_tag["href"]
                product_link = urljoin(url, rel_link)

                price_tag = item.select_one("p.price_color")
                price = price_tag.text.strip() if price_tag else "N/A"

                avail_tag = item.select_one("p.instock.availability")
                availability = " ".join(avail_tag.text.split()) if avail_tag else "N/A"

                rating_p = item.select_one("p.star-rating")
                rating = rating_to_int(rating_p.get("class", [])) if rating_p else None

                results.append({
                    "Title": title,
                    "Price": price,
                    "Availability": availability,
                    "Rating": rating,
                    "Link": product_link
                })
            except Exception as e:
                # skip problematic items but continue
                print("  ⚠ Skipped an item due to:", e)
                continue

        # Find next page link
        next_li = soup.select_one("li.next > a")
        if not next_li:
            break

        next_href = next_li["href"]
        url = urljoin(url, next_href)
        page += 1
        if max_pages and page > max_pages:
            break
        time.sleep(delay)

    # Save to CSV
    if results:
        with open(output_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["Title", "Price", "Availability", "Rating", "Link"])
            writer.writeheader()
            writer.writerows(results)
        print(f"\n✅ Saved {len(results)} books to {output_csv}")
    else:
        print("⚠ No results scraped.")

    return results

if __name__ == "__main__":
    # Example: set max_pages to 2 during dev to speed up testing
    scraped = scrape_books(max_pages=None, delay=0.6, output_csv="books_all.csv")
    # print first 5
    for i, r in enumerate(scraped[:5], start=1):
        print(f"{i}. {r['Title']} — {r['Price']} — Rating: {r['Rating']}")


Scraping page 1: http://books.toscrape.com/
Scraping page 2: http://books.toscrape.com/catalogue/page-2.html
Scraping page 3: http://books.toscrape.com/catalogue/page-3.html
Scraping page 4: http://books.toscrape.com/catalogue/page-4.html
Scraping page 5: http://books.toscrape.com/catalogue/page-5.html
Scraping page 6: http://books.toscrape.com/catalogue/page-6.html
Scraping page 7: http://books.toscrape.com/catalogue/page-7.html
Scraping page 8: http://books.toscrape.com/catalogue/page-8.html
Scraping page 9: http://books.toscrape.com/catalogue/page-9.html
Scraping page 10: http://books.toscrape.com/catalogue/page-10.html
Scraping page 11: http://books.toscrape.com/catalogue/page-11.html
Scraping page 12: http://books.toscrape.com/catalogue/page-12.html
Scraping page 13: http://books.toscrape.com/catalogue/page-13.html
Scraping page 14: http://books.toscrape.com/catalogue/page-14.html
Scraping page 15: http://books.toscrape.com/catalogue/page-15.html
Scraping page 16: http://books.tos